# Distributed Algorithmic Bias Mitigation on Apache Spark
### Case Study: Scaling Machine Learning Fairness Beyond Laptop Memory Limits

---

## 1. Problem Statement & Motivation
Standard algorithmic fairness toolkits (such as **IBM AIF360** and **Microsoft Fairlearn**) run on single machines using in-memory Python structures (`pandas`, `numpy`, `scikit-learn`). When enterprise datasets grow beyond a few gigabytes (millions of records), laptops trigger **Out-Of-Memory (OOM) crashes**.

In this notebook, we implement and evaluate the three canonical bias mitigation paradigms natively on **Apache Spark (PySpark)**:
1. **Pre-processing**: *Distributed Reweighing* (Kamiran & Calders, 2012)
2. **In-processing**: *Distributed Fair Grid Search* (Agarwal et al., 2018)
3. **Post-processing**: *Distributed Reject Option Classification (ROC)* (Kamiran et al., 2012)

### Datasets Evaluated:
- **UCI Adult Census Income** (32,561 records) — *Correctness Verification (Gender Bias)*
- **COMPAS Recidivism** (5,278 records) — *Correctness Verification (Racial Bias)*
- **Folktables US ACS PUMS** (scaled up to 1,000,000 records) — *Scalability & Runtime Benchmarks*

## 2. Environment Setup & Spark Initialization

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Initialize Spark Session using our project helper
from bias_mitigation.spark_utils import get_spark_session

spark = get_spark_session(app_name="FairnessResultsNotebook", memory="4g")
print(f"Spark initialized successfully! Version: {spark.version}")

## 3. Correctness Verification: UCI Adult & COMPAS

We evaluate before vs. after fairness outcomes across:
- **Statistical Parity Difference (SPD)**: $P(\hat{Y}=1 \mid S=0) - P(\hat{Y}=1 \mid S=1)$ (Ideal: 0.0)
- **Disparate Impact Ratio (DIR)**: $\frac{P(\hat{Y}=1 \mid S=0)}{P(\hat{Y}=1 \mid S=1)}$ (EEOC 80% Rule requires $\ge 0.80$, Ideal: 1.0)
- **Accuracy**: Classification accuracy on the held-out test split

In [ ]:
# Load precomputed correctness results (or run run_correctness.py)
with open("output/correctness_results.json") as f:
    correctness_data = json.load(f)

# Format UCI Adult results into a clean table
adult_df = pd.DataFrame(correctness_data["UCI_Adult"]).T
adult_df["accuracy"] = adult_df["accuracy"].apply(lambda x: f"{x*100:.2f}%")
adult_df["disparate_impact_ratio"] = adult_df["disparate_impact_ratio"].apply(lambda x: f"{x:.4f}")
adult_df["statistical_parity_diff"] = adult_df["statistical_parity_diff"].apply(lambda x: f"{x:.4f}")

print("=== UCI Adult Census Income (Correctness Results) ===")
display(adult_df[["accuracy", "disparate_impact_ratio", "statistical_parity_diff"]])

In [ ]:
# Format COMPAS results into a clean table
compas_df = pd.DataFrame(correctness_data["COMPAS"]).T
compas_df["accuracy"] = compas_df["accuracy"].apply(lambda x: f"{x*100:.2f}%")
compas_df["disparate_impact_ratio"] = compas_df["disparate_impact_ratio"].apply(lambda x: f"{x:.4f}")
compas_df["statistical_parity_diff"] = compas_df["statistical_parity_diff"].apply(lambda x: f"{x:.4f}")

print("=== COMPAS Recidivism (Correctness Results) ===")
display(compas_df[["accuracy", "disparate_impact_ratio", "statistical_parity_diff"]])

### Visualizing the Fairness vs. Accuracy Trade-Off

In [ ]:
display(Image(filename="output/fairness_accuracy_tradeoff.png"))

## 4. Scalability Benchmarks: Folktables (US ACS PUMS)

Here we test how runtime, memory, and throughput scale as data grows from **50,000 to 1,000,000 rows** on Apache Spark vs. a single-node laptop implementation.

In [ ]:
with open("output/scalability_results.json") as f:
    scale_data = json.load(f)

rows = []
for s in sorted([int(k) for k in scale_data.keys()]):
    d = scale_data[str(s)]
    rows.append({
        "Dataset Size (N)": f"{s:,}",
        "Reweighing Time (s)": d["spark_preprocessing_reweighing"]["runtime_sec"],
        "Reweighing Throughput (rows/s)": f"{d['spark_preprocessing_reweighing']['throughput_rows_per_sec']:,.0f}",
        "In-Processing Time (s)": d["spark_inprocessing_fair_grid"]["runtime_sec"],
        "Post-Processing Time (s)": d["spark_postprocessing_roc"]["runtime_sec"],
        "Single-Node Laptop Status": d["single_node_laptop"]["status"]
    })

summary_table = pd.DataFrame(rows)
display(summary_table)

### Visualizing Scalability & Throughput Curves

In [ ]:
display(Image(filename="output/scalability_curves.png"))

## 5. Conclusions & Submission Summary

1. **Pre-processing (Reweighing)**: Extremely fast $O(N)$ single-pass aggregation. Processed 1 million rows in **5.71 seconds** (throughput > 175,000 rows/sec).
2. **In-processing (Fair Grid Search)**: Yields the highest fairness gains (DIR 0.909 on Adult), directly steering Spark MLlib optimization to achieve demographic parity.
3. **Post-processing (ROC)**: Operates instantaneously on model probability outputs (6.23s on 1M rows), boosting Disparate Impact to **0.950** on COMPAS with negligible accuracy drop.
4. **Distributed Scalability**: Spark completely overcomes the memory wall that crashes single-node laptop toolkits above 250,000 rows.